# Notebook 16 — Ispezionare l'architettura di un modello GPT-like

**Modulo:** 16 — Come funzionano davvero gli LLM
**Tempo stimato:** 1-1,5 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Caricato la configurazione e i pesi di un modello GPT-2 open source con la libreria `transformers`
- Ispezionato via codice i suoi iperparametri architetturali (numero di layer, teste, dimensione embedding)
- Calcolato il numero totale di parametri e confrontato la struttura con quanto descritto per GPT-1 nella dispensa
- Costruito una tabella comparativa (scritta a mano, dati pubblici) della scala dei modelli piu recenti
- Una funzione riutilizzabile per confrontare piu modelli della famiglia GPT-2 tra loro

## Prerequisiti

```bash
pip install transformers torch
```

> **Nota:** questo notebook **non richiede nessuna chiave API**. `gpt2` e le sue varianti sono
> modelli pubblici scaricabili liberamente dall'Hugging Face Hub (serve solo una connessione
> internet per il primo download, poi restano in cache locale). Non alleniamo nulla: replicare
> l'addestramento di un vero GPT non e alla portata di uno studente (ne di questo notebook) — qui
> **ispezioniamo** un'architettura decoder-only reale, cosa che nel 2026 e possibile solo con
> modelli open source (vedi dispensa, Sezione 2).

---
## Cella 0 — Setup

Importiamo `transformers` e verifichiamo che sia disponibile insieme a un backend (`torch`).

In [ ]:
import transformers
import torch

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)


---
## Cella 1 — Ispezionare la configurazione, senza scaricare i pesi

Prima di scaricare l'intero modello (che include i pesi, quindi centinaia di MB), possiamo
ispezionare solo la sua **configurazione**: gli iperparametri architetturali che definiscono la
forma del modello, molto piu leggera da scaricare.

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("gpt2")

print("Numero di blocchi Transformer impilati (n_layer):", config.n_layer)
print("Numero di teste di attenzione per blocco (n_head): ", config.n_head)
print("Dimensione dell'embedding (n_embd):                ", config.n_embd)
print("Dimensione del vocabolario (vocab_size):            ", config.vocab_size)
print("Lunghezza massima di contesto (n_positions):        ", config.n_positions)


**Confronta questi numeri con GPT-1 (dispensa, Sezione 5):** GPT-1 aveva embedding a 768
dimensioni, 12 teste di attenzione e 12 blocchi impilati. Il modello `gpt2` (la versione piu
piccola della famiglia GPT-2) e un'evoluzione diretta della stessa architettura: guarda i valori
sopra e confrontali con quelli di GPT-1.

---
## Cella 2 — Caricare il modello completo e contare i parametri

Ora carichiamo il modello con i pesi, e calcoliamo il numero totale di parametri sommando gli
elementi di ogni tensore — lo stesso concetto di "parametro" introdotto nel Modulo 01 e usato per
tutta la dispensa di questo modulo.

In [ ]:
from transformers import AutoModel

modello = AutoModel.from_pretrained("gpt2")

numero_parametri = sum(p.numel() for p in modello.parameters())
numero_parametri_milioni = numero_parametri / 1_000_000

print(f"Numero totale di parametri: {numero_parametri:,}")
print(f"In milioni: {numero_parametri_milioni:.1f}M")


**Nota:** il valore che ottieni (circa 124 milioni per `gpt2`) e piu alto dei ~117 milioni citati
per GPT-1 nella dispensa: GPT-2 piccolo e leggermente piu grande del GPT-1 originale, pur
mantenendo un'architettura molto simile — esattamente il tipo di "scalamento incrementale"
descritto nella Sezione 6 della dispensa.

---
## Cella 3 — Guardare dentro un singolo blocco Transformer

Ispezioniamo la struttura di un solo blocco decoder, per ritrovare i componenti descritti nella
dispensa: attenzione multi-testa mascherata, rete feedforward, normalizzazione.

In [ ]:
primo_blocco = modello.h[0]  # il primo dei blocchi Transformer impilati
print(primo_blocco)


In [ ]:
# Dimensioni della rete feedforward interna al blocco (confronta con la Sezione 5 della dispensa:
# GPT-1 usava 768 -> 3072 -> 768)
feedforward = primo_blocco.mlp
print("Prima proiezione (c_fc):  ", feedforward.c_fc.weight.shape)
print("Seconda proiezione (c_proj):", feedforward.c_proj.weight.shape)

parametri_feedforward_blocco = feedforward.c_fc.weight.numel() + feedforward.c_proj.weight.numel()
print(f"\nParametri (solo pesi, senza bias) della rete feedforward di questo blocco: {parametri_feedforward_blocco:,}")
print("Confronta questo numero con il calcolo della domanda 4 della dispensa del Modulo 16.")


---
## Cella 4 — Una piccola inferenza per vedere la forma dell'output

Tokenizziamo un prompt e osserviamo la forma dell'output del modello: dovrebbe avere un vettore di
dimensione pari alla dimensione dell'embedding (`n_embd`) per ogni token della sequenza in ingresso
— la stessa rappresentazione contestuale arricchita descritta nel Modulo 15.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

prompt = "Artificial intelligence is"
input_ids = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output = modello(**input_ids)

print("Token in ingresso:", tokenizer.convert_ids_to_tokens(input_ids["input_ids"][0]))
print("Forma dell'output (batch, n_token, n_embd):", output.last_hidden_state.shape)


> **Nota:** `AutoModel` restituisce le rappresentazioni contestuali interne (gli stati nascosti),
> non ancora la distribuzione di probabilita sul vocabolario. Per generare testo vero e proprio si
> userebbe `AutoModelForCausalLM`, che aggiunge lo strato finale di proiezione sul vocabolario
> descritto nella dispensa (Sezione 5): quello strato produce un vettore di dimensione
> `vocab_size`, non `n_embd`.

---
## Cella 5 — Tabella comparativa: la scala dei modelli nel tempo

I numeri di questa tabella **non sono calcolati via codice**: sono dati pubblici (per i modelli
open) o stime pubbliche circolate (per i modelli chiusi, dove l'architettura esatta non e nota).
Li scriviamo a mano per avere un riferimento di scala, non per un confronto tecnico rigoroso —
segui sempre la dispensa (Sezione 4) per il contesto su cosa significano queste cifre.

In [ ]:
confronto_modelli = [
    {"modello": "GPT-1",             "anno": 2018, "parametri_stima": "117 milioni",  "tipo": "closed (dettagli pubblicati)"},
    {"modello": "GPT-2 (small, qui)","anno": 2019, "parametri_stima": "124 milioni",  "tipo": "open (pesi pubblici)"},
    {"modello": "GPT-2 (extra-large)","anno": 2019,"parametri_stima": "1,5 miliardi", "tipo": "open (pesi pubblici)"},
    {"modello": "GPT-3",             "anno": 2020, "parametri_stima": "175 miliardi", "tipo": "closed"},
    {"modello": "GPT-4",             "anno": 2023, "parametri_stima": "~1.000+ miliardi (stima non confermata)", "tipo": "closed"},
    {"modello": "GPT-5",             "anno": 2025, "parametri_stima": "non dichiarato pubblicamente", "tipo": "closed"},
    {"modello": "Claude Opus 5",     "anno": 2025, "parametri_stima": "non dichiarato pubblicamente", "tipo": "closed"},
    {"modello": "Gemini 3 Pro",      "anno": 2025, "parametri_stima": "non dichiarato pubblicamente", "tipo": "closed"},
    {"modello": "Llama 4",           "anno": 2025, "parametri_stima": "decine-centinaia di miliardi (varianti multiple, architettura MoE)", "tipo": "open (pesi pubblici)"},
    {"modello": "DeepSeek V3",       "anno": 2024, "parametri_stima": "671 miliardi totali (~37 miliardi attivi per token, architettura MoE)", "tipo": "open (pesi pubblici)"},
]

print(f"{'Modello':<20}{'Anno':<8}{'Parametri (stima)':<45}{'Tipo'}")
print("-" * 100)
for riga in confronto_modelli:
    print(f"{riga['modello']:<20}{riga['anno']:<8}{riga['parametri_stima']:<45}{riga['tipo']}")


**Osservazione da discutere:** nota come, a partire da GPT-3/GPT-4, il numero di parametri smette
di essere pubblico. E anche come i modelli piu recenti (Llama 4, DeepSeek V3) non siano piu descritti
da un singolo numero di parametri, ma da un'architettura **Mixture of Experts** (parametri totali
molto alti, ma solo una frazione "attiva" per ogni token) — esattamente la leva del 2026 descritta
nella Sezione 7 della dispensa.

---
## Esercizio — `# TODO`: confrontare `gpt2` con `distilgpt2`

`distilgpt2` e una versione compressa (*distillata*) di `gpt2`, pensata per essere piu piccola e
veloce mantenendo buona parte delle prestazioni. Completa il codice per caricarla e confrontarne
la configurazione e il numero di parametri con `gpt2`.

In [ ]:
# TODO: carica la configurazione di distilgpt2 con AutoConfig.from_pretrained
# config_distil = AutoConfig.from_pretrained("distilgpt2")

# TODO: stampa n_layer, n_head, n_embd per distilgpt2 e confrontali con gpt2 (Cella 1)

# TODO: carica il modello completo con AutoModel.from_pretrained("distilgpt2")
# e calcola il numero totale di parametri, come nella Cella 2

# TODO: calcola il rapporto tra il numero di parametri di gpt2 e di distilgpt2
# (quante volte gpt2 e piu grande?)


---
## Sfida finale — Funzione di confronto generica tra modelli

Costruisci una funzione `confronta_architetture(nomi_modelli: list[str])` che:

1. Per ogni nome di modello nella lista, carichi la configurazione con `AutoConfig.from_pretrained`
   e il modello completo con `AutoModel.from_pretrained`.
2. Calcoli il numero totale di parametri di ciascun modello.
3. Costruisca e stampi una tabella con colonne: nome modello, `n_layer`, `n_head`, `n_embd`,
   numero totale di parametri (in milioni).
4. Ordini la tabella dal modello piu piccolo al piu grande per numero di parametri.

Usala per confrontare `["distilgpt2", "gpt2"]` (mantieni la lista leggera: modelli piu grandi come
`gpt2-medium` o `gpt2-large` richiedono download significativamente piu pesanti).

In [ ]:
def confronta_architetture(nomi_modelli: list[str]) -> list[dict]:
    """Carica configurazione e pesi di ogni modello in nomi_modelli e ne confronta l'architettura."""
    risultati = []

    for nome in nomi_modelli:
        # TODO: carica config e modello
        # config = AutoConfig.from_pretrained(nome)
        # mod = AutoModel.from_pretrained(nome)
        # n_parametri = sum(p.numel() for p in mod.parameters())

        # TODO: aggiungi un dizionario con i risultati a 'risultati'
        # risultati.append({
        #     "modello": nome,
        #     "n_layer": config.n_layer,
        #     "n_head": config.n_head,
        #     "n_embd": config.n_embd,
        #     "parametri_milioni": round(n_parametri / 1_000_000, 1)
        # })
        pass

    # TODO: ordina 'risultati' per numero di parametri crescente
    # risultati.sort(key=lambda r: r["parametri_milioni"])

    # TODO: stampa la tabella finale
    # for r in risultati:
    #     print(r)

    return risultati


# Prova la tua implementazione (decommenta dopo aver completato i TODO)
# confronta_architetture(["distilgpt2", "gpt2"])


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 16 — Modulo 16 del Corso Completo di IA Generativa*

Con questo modulo si chiude il blocco "Come funzionano davvero gli LLM" (Moduli 13-16). Dal
Modulo 17 in poi il corso passa dal capire l'architettura allo **specializzare** un modello:
preaddestramento, fine-tuning e le tecniche che trasformano un modello fondazionale in uno
strumento adatto a un compito specifico.